# Siapkan Dataset Two-Stage -> Google Drive

Notebook ini membangun folder `/content/datasets` berisi **8 kelas daun pisang + 1 kelas negatif `Not Banana Leaf`**.

Dataset ini dipakai untuk pipeline two-stage:
1. **Banana Gate**: `Banana Leaf` vs `Not Banana Leaf`.
2. **Disease Classifier**: hanya 8 kelas daun pisang.

Output akhir adalah `banana_datasets_twostage.zip` di Google Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Kredensial Kaggle

In [ ]:
import os

DRIVE_KAGGLE = '/content/drive/MyDrive/kaggle.json'
if os.path.exists(DRIVE_KAGGLE):
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp "{DRIVE_KAGGLE}" /root/.kaggle/kaggle.json
else:
    from google.colab import files
    files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp kaggle.json /root/.kaggle/kaggle.json

!chmod 600 /root/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle --version

## 3. Konfigurasi

In [ ]:
from pathlib import Path

STAGING = Path('/content/.staging')
DATASETS = Path('/content/datasets')
DRIVE_OUT = Path('/content/drive/MyDrive/banana-datasets')
DRIVE_HARD_NEGATIVES = DRIVE_OUT / 'hard_negatives'
SEED = 42
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Kuota negatif Kaggle. hard_negatives dari Drive selalu ditambahkan di luar kuota.
NEG_TOTAL = 4000

BANANA_LABELS = [
    'Augmented Banana Black Sigatoka Disease',
    'Augmented Banana Bract Mosaic Virus Disease',
    'Augmented Banana Cordana Disease',
    'Augmented Banana Healthy Leaf',
    'Augmented Banana Insect Pest Disease',
    'Augmented Banana Moko Disease',
    'Augmented Banana Panama Disease',
    'Augmented Banana Yellow Sigatoka Disease',
]
NEG_LABEL = 'Not Banana Leaf'

BANANA_SOURCES = {
    'main': 'sujaykapadnis/banana-disease-recognition-dataset',
    'cordana': 'shifatearman/bananalsd',
}

# Fokus bahan uji: pepaya + kelapa dominan. Sumber lain tetap ada sebagai pendukung.
NEG_SOURCES = [
    {'slug': 'ajithdari/papaya-leaf-disease-dataset',                     'weight': 0.35, 'group': 'daun-pepaya'},
    {'slug': 'shravanatirtha/coconut-leaf-dataset-for-pest-identification','weight': 0.35, 'group': 'daun-kelapa'},
    {'slug': 'nirmalsankalana/plantdoc-dataset',                          'weight': 0.10, 'group': 'daun-multispesies'},
    {'slug': 'shyambhu/hands-and-palm-images-dataset',                    'weight': 0.08, 'group': 'tangan'},
    {'slug': 'prasunroy/natural-images',                                  'weight': 0.07, 'group': 'objek-acak'},
    {'slug': 'itsahmad/indoor-scenes-cvpr-2019',                          'weight': 0.05, 'group': 'scene-lantai'},
]

## 4. Fungsi Bantu

In [ ]:
import random, shutil, subprocess

def kaggle_download(slug, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    if any(dest.iterdir()):
        print(f'  [skip] {slug} (sudah ada)')
        return True
    print(f'  [download] {slug}')
    try:
        subprocess.run(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(dest), '--unzip'], check=True)
        return True
    except subprocess.CalledProcessError as e:
        print(f'  [GAGAL] {slug}: {e}')
        return False

def list_images(root):
    root = Path(root)
    return [p for p in root.rglob('*') if p.suffix.lower() in IMG_EXT and p.is_file()]

def norm(s):
    return ''.join(c for c in s.lower() if c.isalnum())

def find_class_dir(staging, label):
    key = label.replace('Augmented Banana', '').replace('Disease', '').replace('Leaf', '').strip()
    key_n = norm(key); cands = []
    for d in Path(staging).rglob('*'):
        if not d.is_dir(): continue
        dn = norm(d.name)
        if not dn: continue
        if dn == norm(label): cands.append((3, d))
        elif key_n and key_n in dn: cands.append((2, d))
        elif key_n and dn in key_n and len(dn) >= 4: cands.append((1, d))
    if not cands: return None
    cands.sort(key=lambda t: (t[0], len(list_images(t[1]))), reverse=True)
    return cands[0][1]

def copy_sample(images, n, dest, prefix, rng):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    chosen = images if len(images) <= n else rng.sample(images, n)
    for i, src in enumerate(chosen):
        try:
            shutil.copy2(src, dest / f'{prefix}_{i:05d}{src.suffix.lower()}')
        except OSError:
            pass
    return len(chosen)

## 5. Bangun 8 Kelas Daun Pisang

In [ ]:
print('=== Tahap 1: pisang ===')
main_stage = STAGING / 'banana_main'
cordana_stage = STAGING / 'banana_cordana'
kaggle_download(BANANA_SOURCES['main'], main_stage)
kaggle_download(BANANA_SOURCES['cordana'], cordana_stage)

banana_counts = {}
for label in BANANA_LABELS:
    search_root = cordana_stage if 'Cordana' in label else main_stage
    src_dir = find_class_dir(search_root, label)
    dest = DATASETS / label
    dest.mkdir(parents=True, exist_ok=True)
    if src_dir is None:
        print(f'  [!] sumber untuk {label!r} TIDAK ditemukan')
        banana_counts[label] = len(list_images(dest))
        continue
    rng = random.Random(SEED)
    imgs = list_images(src_dir)
    copy_sample(imgs, len(imgs), dest, norm(label)[:12], rng)
    banana_counts[label] = len(list_images(dest))
    print(f'  [{label}] <- {src_dir.name} ({banana_counts[label]})')

## 6. Bangun Kelas Negatif

In [ ]:
print(f'=== Tahap 2: negatif Kaggle (target {NEG_TOTAL}) ===')
dest = DATASETS / NEG_LABEL
if dest.exists():
    shutil.rmtree(dest)
dest.mkdir(parents=True, exist_ok=True)

active = [s for s in NEG_SOURCES if s.get('enabled', True)]
total_w = sum(s['weight'] for s in active)
for s in active:
    stage = STAGING / ('neg_' + s['group'])
    if not kaggle_download(s['slug'], stage):
        print(f"  [lewati] {s['group']}")
        continue
    imgs = list_images(stage)
    if not imgs:
        print(f"  [lewati] {s['group']} (kosong)")
        continue
    quota = max(1, round(NEG_TOTAL * s['weight'] / total_w))
    rng = random.Random(SEED + hash(s['group']) % 1000)
    n = copy_sample(imgs, quota, dest, s['group'], rng)
    print(f"  [{s['group']:18s}] tersedia {len(imgs):5d} -> diambil {n} (kuota {quota})")

hard_imgs = list_images(DRIVE_HARD_NEGATIVES) if DRIVE_HARD_NEGATIVES.exists() else []
if hard_imgs:
    rng = random.Random(SEED + 9999)
    n = copy_sample(hard_imgs, len(hard_imgs), dest, 'hardnegative', rng)
    print(f"  [{'hard-negatives':18s}] tersedia {len(hard_imgs):5d} -> ditambahkan {n} (di luar kuota)")
else:
    print(f'  [hard-negatives] kosong/tidak ada: {DRIVE_HARD_NEGATIVES}')

print('  TOTAL negatif:', len(list_images(dest)))

## 7. Ringkasan

In [ ]:
print('=== Ringkasan datasets/ ===')
grand = 0
for label in BANANA_LABELS + [NEG_LABEL]:
    n = len(list_images(DATASETS / label)); grand += n
    print(f'  {label:45s} : {n}')
print(f'  {"TOTAL":45s} : {grand}')

## 8. Zip -> Simpan ke Drive

In [ ]:
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
zip_base = '/content/banana_datasets_twostage'
print('Membuat zip... (bisa beberapa menit)')
shutil.make_archive(zip_base, 'zip', root_dir=str(DATASETS))
zip_path = zip_base + '.zip'
size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zip dibuat: {zip_path} ({size_mb:.0f} MB)')

dst = DRIVE_OUT / 'banana_datasets_twostage.zip'
!cp "{zip_path}" "{dst}"
print('Tersimpan di Drive:', dst)
print('\nDi notebook training, ekstrak dengan:')
print(f'  !cp "{dst}" /content/ && unzip -q /content/banana_datasets_twostage.zip -d /content/datasets')